## EXPLORATION — REQUÊTAGE DES TABLES ICEBERG

In [ ]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

#LORSQU'ON TRAVAILLE DEPUIS SA MACHINE LOCAL
MINIO_ENDPOINT = "http://192.168.1.230:30137"
MINIO_ACCESS_KEY = "datalab-team"
MINIO_SECRET_KEY = "minio-datalabteam123"
NESSIE_URI = "http://192.168.1.230:30604/api/v1"

#---------------------------------------------------------------------------------

#LORSQU'ON TRAVAILLE SUR JHUB
# MINIO_ENDPOINT = "http://minio.mon-namespace.svc.cluster.local:80"
# MINIO_ACCESS_KEY = "datalab-team"
# MINIO_SECRET_KEY = "minio-datalabteam123"
# NESSIE_URI = "http://nessie.trino.svc.cluster.local:19120/api/v1"

#---------------------------------------------------------------------------------

conf = (
    SparkConf()
    .setAppName("Exploration")
    .set("spark.driver.host", "127.0.0.1")
    .set("spark.driver.bindAddress", "127.0.0.1")
    .set("spark.driver.memory", "16g")
    .set("spark.jars.packages",
         "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
         "org.apache.hadoop:hadoop-aws:3.3.4,"
         "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1")

    .set("spark.sql.extensions",
         "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
         "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")

    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    .set("spark.sql.catalog.nessie.warehouse", "s3a://bronze/")

    .set("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .set("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .set("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)

In [ ]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [ ]:
# Lister toutes les tables disponibles dans le catalogue
spark.sql("SHOW NAMESPACES IN nessie").show()
spark.sql("SHOW TABLES IN nessie.bronze").show()
spark.sql("SHOW TABLES IN nessie.silver").show()

## ON EFFECTUE LES REQUÊTES ET ANALYSES ICI

In [ ]:
# Lecture d'une table
df = spark.table("nessie.silver.<<nom_de_la_table>>")
print(f"{df.count():,} lignes × {len(df.columns)} colonnes")
df.printSchema()

In [ ]:
# Requête SQL directe sur Iceberg
resultat = spark.sql("""
    SELECT
        <<colonne_groupe>>,
        COUNT(*)            AS nb_lignes,
        AVG(<<montant>>)    AS moyenne,
        SUM(<<montant>>)    AS total
    FROM nessie.silver.<<nom_de_la_table>>
    WHERE <<condition>>
    GROUP BY <<colonne_groupe>>
    ORDER BY total DESC
""")
resultat.show(20)

In [ ]:
# Conversion vers pandas pour visualisation (sur un sous-ensemble filtré)
# Ne pas faire .toPandas() sur toute une grosse table !
df_pandas = resultat.toPandas()
df_pandas.head(10)

In [ ]:
# Time travel Iceberg : lire une version passée de la table
# spark.read.option("as-of-timestamp", "2024-01-01 00:00:00").table("nessie.silver.ma_table")
# spark.read.option("snapshot-id", "<snapshot_id>").table("nessie.silver.ma_table")

# Historique des snapshots
# spark.sql("SELECT * FROM nessie.silver.ma_table.snapshots").show()

In [ ]:
# Export des résultats vers MinIO (staging ou gold)
# resultat.write \
#     .format("iceberg") \
#     .mode("overwrite") \
#     .saveAsTable("nessie.gold.<<nom_resultat>>")

# Ou export CSV vers staging pour partage
# resultat.coalesce(1).write \
#     .option("header", "true") \
#     .csv("s3a://staging/exports/<<nom_resultat>>/")